# Week 2 — Exploratory Data Analysis and Visualization
**Student Name:** Sindhu Patil | **Internship:** Data Science

## 1. Introduction
Exploratory Data Analysis (EDA) is a fundamental phase in data science that uncovers underlying statistical distributions, identifies feature correlations, evaluates class balance, and extracts domain-specific business insights before modeling.

## 2. Objective
Perform data-backed EDA on the cleaned IBM Telco Customer Churn dataset (`data/processed/cleaned_telco_churn.csv`). Quantify relationships between customer demographics, contract commitments, payment channels, service add-ons, and churn rates.

## 3. Import Libraries & Configuration

In [ ]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.visualization import (
    plot_churn_target_distribution, plot_churn_by_contract, plot_churn_by_internet_service,
    plot_churn_by_payment_method, plot_tenure_distribution, plot_monthly_charges_by_churn,
    plot_total_charges_by_churn, plot_correlation_heatmap, plot_multivariate_contract_charges,
    plot_demographics_churn, plot_services_churn
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('husl')

## 4. Load Cleaned Dataset

In [ ]:
data_path = '../data/processed/cleaned_telco_churn.csv'
df = pd.read_csv(data_path)
print('Loaded Dataset Shape:', df.shape)
print('Missing values:', df.isnull().sum().sum())
print('Duplicate rows:', df.duplicated().sum())

## 5. Dataset Overview & Data Structure

In [ ]:
print('Columns:', df.columns.tolist())
df.info()

In [ ]:
df.head(5)

In [ ]:
df.tail(5)

## 6. Descriptive Statistics
Calculating Mean, Median, Standard Deviation, Min, Max, Q1, Q3, and IQR for numerical features.

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
stats = df[num_cols].describe().T
stats['median'] = df[num_cols].median()
stats['iqr'] = stats['75%'] - stats['25%']
stats = stats[['mean', 'std', 'min', '25%', 'median', '75%', 'max', 'iqr']]
stats

## 7. Target Variable — Churn Analysis

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pcts = df['Churn'].value_counts(normalize=True) * 100
churn_summary = pd.DataFrame({'Customer_Count': churn_counts, 'Percentage_%': churn_pcts})
print(churn_summary)

fig1 = plot_churn_target_distribution(df, 'churn_distribution.png')
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Churn', palette=['#2ecc71', '#e74c3c'])
plt.title('Target Churn Class Distribution (73.5% vs 26.5%)')
plt.show()

## 8. Univariate Feature Analysis
Visualizing individual feature distributions.

In [ ]:
plot_tenure_distribution(df, 'tenure_distribution.png')
plt.figure(figsize=(8, 4))
sns.histplot(df['tenure'], kde=True, color='#3498db')
plt.title('Univariate Tenure Distribution (Months)')
plt.show()

In [ ]:
plot_monthly_charges_by_churn(df, 'monthly_charges_by_churn.png')
plot_total_charges_by_churn(df, 'total_charges_by_churn.png')

## 9. Bivariate Analysis & Churn Rates by Category

In [ ]:
contract_ct = pd.crosstab(df['Contract'], df['Churn'], margins=True)
contract_ct['Churn_Rate_%'] = (pd.crosstab(df['Contract'], df['Churn'], normalize='index')['Yes'] * 100).round(2)
print('--- Contract Churn Rates ---')
print(contract_ct)
plot_churn_by_contract(df, 'churn_by_contract.png')

In [ ]:
internet_ct = pd.crosstab(df['InternetService'], df['Churn'], margins=True)
internet_ct['Churn_Rate_%'] = (pd.crosstab(df['InternetService'], df['Churn'], normalize='index')['Yes'] * 100).round(2)
print('--- Internet Service Churn Rates ---')
print(internet_ct)
plot_churn_by_internet_service(df, 'churn_by_internet_service.png')

In [ ]:
pay_ct = pd.crosstab(df['PaymentMethod'], df['Churn'], margins=True)
pay_ct['Churn_Rate_%'] = (pd.crosstab(df['PaymentMethod'], df['Churn'], normalize='index')['Yes'] * 100).round(2)
print('--- Payment Method Churn Rates ---')
print(pay_ct)
plot_churn_by_payment_method(df, 'churn_by_payment_method.png')

## 10. Tenure Cohort Analysis

In [ ]:
df['Tenure_Cohort'] = pd.cut(df['tenure'], bins=[-1, 12, 24, 48, 60, 72], labels=['0-12 mos', '13-24 mos', '25-48 mos', '49-60 mos', '61-72 mos'])
tenure_ct = pd.crosstab(df['Tenure_Cohort'], df['Churn'], margins=True)
tenure_ct['Churn_Rate_%'] = (pd.crosstab(df['Tenure_Cohort'], df['Churn'], normalize='index')['Yes'] * 100).round(2)
print('--- Tenure Cohort Churn Rates ---')
print(tenure_ct)

## 11. Demographic & Value-Added Services Analysis

In [ ]:
plot_demographics_churn(df, 'demographics_churn.png')
plot_services_churn(df, 'services_churn.png')

## 12. Multivariate Analysis

In [ ]:
plot_multivariate_contract_charges(df, 'multivariate_contract_charges.png')
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x='Contract', y='MonthlyCharges', hue='Churn', palette=['#2ecc71', '#e74c3c'])
plt.title('Monthly Charges by Contract & Churn Status')
plt.show()

## 13. Correlation Analysis

In [ ]:
plot_correlation_heatmap(df, 'correlation_heatmap.png')
num_corr = df[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
num_corr['Churn_Numeric'] = (df['Churn'] == 'Yes').astype(int)
print('Correlations with Churn:')
print(num_corr.corr()['Churn_Numeric'])

## 14. Key Findings
1. Target Imbalance: 26.54% churn rate (1,869 churners vs 5,174 retained).
2. Contract Type: Month-to-Month contracts have 42.71% churn rate vs 2.83% for 2-Year plans.
3. Internet Service: Fiber Optic subscribers exhibit 41.89% churn rate vs 18.96% for DSL.
4. Payment Method: Electronic Check users show the highest churn rate at 45.29%.
5. Tenure Impact: 0–12 Month subscribers have 47.44% churn rate vs 6.61% for 61–72 Month tenure.
6. Security Add-ons: Customers without Online Security have 41.77% churn rate vs 14.61% with security.
7. Tech Support: Customers without Tech Support have 41.64% churn rate vs 15.17% with support.
8. Senior Status: Senior citizens churn at 41.68% vs 23.61% for non-seniors.
9. Correlation: Tenure is negatively correlated with churn (r = -0.352).
10. Monthly Charges: Higher monthly bill (mean $74.44 for churners vs $61.27 for retained) increases churn risk.

## 15. Business Insights & Recommendations
- DATA FINDING: 42.71% Month-to-Month churn vs 2.83% 2-Year contract churn.
  - BUSINESS STRATEGY: Offer a 15% annual contract renewal discount.
- DATA FINDING: 41.89% Fiber Optic churn rate with mean monthly bill of $91.50.
  - BUSINESS STRATEGY: Bundle free Online Security and Tech Support for Fiber subscribers.
- DATA FINDING: 45.29% Electronic Check churn rate.
  - BUSINESS STRATEGY: Incentive automated payment methods (Auto Credit Card / ACH) with a $5 monthly bill credit.

## 16. Technical Challenges & Solutions
- Class Imbalance: Handled by focusing on Recall, F1-Score, and ROC-AUC rather than raw Accuracy.
- Pandas 3.0 String Types: Explicitly cast boolean masks and integer categorical indicators.

## 17. Limitations
- Cross-sectional historical data lacks real-time time-series telemetry.
- Observational data proves correlation, not direct causality.

## 18. Conclusion
Week 2 EDA identified Month-to-Month contracts, Fiber Optic service, Electronic Check payments, and early tenure (<12 months) as primary drivers of customer churn.